# Data Analysis & Modeling: Will a Player Return?

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
The format follows the interview description in your [shared conversation](https://chatgpt.com/share/6ad92cab-a9dc-83e8-b72e-654063228d54): **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic; do not interpret these results as Roblox player behavior.

## The interviewer prompt

You work on a game platform. At the start of day 0, predict whether each player will return during days 0 through 6, using only information available before day 0.

**Problem 1 — Data analysis (14 minutes).** Load two CSVs, inspect their quality, clean the session log, and build one row per player. Report the percentage of players with no historical sessions and average historical play time per device, including inactive players.

**Problem 2 — Modeling (21 minutes).** Select valid features, create a holdout set, train a simple classification baseline, evaluate it against a no-skill baseline, and predict for two new players. Explain your choices as you code. No deep learning or from-scratch optimizer is required.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the session log |
| 9–14 min | Aggregate and join |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? Which mistake is more costly?


**Reference version.** Run cells top to bottom. Code is complete; each step includes checks and expandable follow-up answers. Try codesignal_roblox_retention_practice.ipynb first if you want a timed attempt.


## Data contract — supplied by the interviewer

**players.csv:** one row per player; attributes are snapshots taken before day 0. All players' outcome windows have completed, so labels are known. Every player belongs in the analysis, even if they have no session history.

| Column | Meaning |
| --- | --- |
| user_id | Unique identifier; not a numeric predictor |
| account_age_days | Account age at the cutoff; may be missing |
| country, device | Categorical attributes at the cutoff; may be missing |
| returned_next_week | Label: 1 if the player returns during days 0–6, otherwise 0 |
| next_week_minutes | Total minutes during days 0–6; present in the offline export, **unavailable at prediction time** |

**sessions.csv:** one row per logged session before cleaning.

| Column | Meaning |
| --- | --- |
| session_id | Session identifier; repeated exports may contain exact duplicate rows |
| user_id | Player identifier, always referencing a supplied player |
| day_offset | Relative to the cutoff: only **-7 through -1** are valid historical features |
| minutes_played | Session duration; blank or a negative sentinel means unknown, not zero |
| completed | 0/1 session-completion indicator; valid even if duration is unknown |

All session rows are available in this small batch exercise; the export deliberately contains some post-cutoff rows. For this mock, distinct players are assumed to be independent samples from the same population. A future-deployment evaluation would instead need multiple cutoffs and a chronological holdout.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter. Select the Python environment at **.venv-codesignal/bin/python** (or another environment with pandas, NumPy and scikit-learn). The two CSVs live in **data/codesignal_roblox/**. No network or GPU is used during the exercise. For a fresh local setup, see **CODESIGNAL_PRACTICE.md**; the real CodeSignal environment has its own libraries.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

SEED = 42
DATA_DIR = Path("data/codesignal_roblox")
assert (DATA_DIR / "players.csv").exists(), "Run from the workspace root, or change DATA_DIR to your CSV folder."
assert (DATA_DIR / "sessions.csv").exists()


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Load **players** and **sessions**. Inspect shapes, sample rows, dtypes, missingness and numeric summaries. Check player uniqueness, duplicate sessions, invalid durations, out-of-window records, and label balance. Do not start dropping every incomplete row.

**Say before coding:** "I will first check the unit of observation and data quality. The label is at player level, but the event log has many rows per player."

**Possible interviewer questions**

1. What does one training example represent?
2. Which columns must not become features?
3. What do you check before filling missing values?


In [2]:
players = pd.read_csv(DATA_DIR / "players.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")

print("players:", players.shape, "sessions:", sessions.shape)
display(players.head(), sessions.head())
players.info()
sessions.info()
display(players.isna().sum(), sessions.isna().sum())
display(sessions.describe())
display(players["returned_next_week"].value_counts(normalize=True).rename("fraction"))
print("duplicate player IDs:", players["user_id"].duplicated().sum())
print("exact duplicate sessions:", sessions.duplicated().sum())
print("negative durations:", sessions["minutes_played"].lt(0).sum())
print("outside history window:", (~sessions["day_offset"].between(-7, -1)).sum())


players: (1000, 6) sessions: (4965, 5)


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes
0,user_0001,184.0,US,mobile,0,0.00
1,user_0002,6.0,US,mobile,0,0.00
2,user_0003,56.0,US,desktop,0,0.00
3,user_0004,39.0,US,desktop,1,247.43
4,user_0005,295.0,GB,NaN,1,5.96


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             1000 non-null   object 
 1   account_age_days    970 non-null    float64
 2   country             950 non-null    object 
 3   device              950 non-null    object 
 4   returned_next_week  1000 non-null   int64  
 5   next_week_minutes   1000 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 47.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4965 entries, 0 to 4964
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   session_id      4965 non-null   object 
 1   user_id         4965 non-null   object 
 2   day_offset      4965 non-null   int64  
 3   minutes_played  4866 non-null   float64
 4   completed       4965 non-null   int64  
dtypes: float64(1),

user_id                0
account_age_days      30
country               50
device                50
returned_next_week     0
next_week_minutes      0
dtype: int64

session_id         0
user_id            0
day_offset         0
minutes_played    99
completed          0
dtype: int64

,day_offset,minutes_played,completed
count,4965.000000,4866.000000,4965.000000
mean,-3.978651,35.123428,0.559718
std,2.051008,29.151789,0.496471
min,-7.000000,-1.000000,0.000000
25%,-6.000000,16.312500,0.000000
50%,-4.000000,27.085000,1.000000
75%,-2.000000,44.290000,1.000000
max,3.000000,240.000000,1.000000


returned_next_week
0    0.707
1    0.293
Name: fraction, dtype: float64

duplicate player IDs: 0
exact duplicate sessions: 30
negative durations: 49
outside history window: 30


In [3]:
assert players["user_id"].is_unique
assert players["returned_next_week"].notna().all()
assert set(players["returned_next_week"]) == {0, 1}
assert sessions["user_id"].isin(players["user_id"]).all()
print("Input contract checked.")


Input contract checked.


<details>
<summary>Follow-up answers</summary>

**1. What does one training example represent?**

One player at one prediction cutoff, not one session. Session data must be aggregated before modeling.

**2. Which columns must not become features?**

The label, user_id, next_week_minutes, and any post-cutoff activity. The identifier is not meaningful ordered data, and the future fields leak the outcome.

**3. What do you check before filling missing values?**

The meaning of missingness, its frequency, and whether it means unknown or no activity. A missing duration is not evidence of a zero-minute session.

</details>


## Problem 1 · Step 2 — Clean the session log (4–9 min)

Create **clean_sessions** without modifying **sessions**. Remove exact duplicate rows, keep only day offsets -7 through -1, and convert negative durations to missing values. Keep a session with an unknown duration: it still counts as activity. Fail visibly if duplicate session IDs remain with conflicting rows.

**Say before coding:** "I will apply deterministic validity rules now. I will postpone learned imputation statistics until after the train/test split."

**Possible interviewer questions**

1. Why not use dropna() on the whole table?
2. Why drop exact duplicates rather than silently keep the first session ID?
3. Is this cleaning before splitting a leak?


In [4]:
clean_sessions = sessions.drop_duplicates().copy()
clean_sessions = clean_sessions.loc[
    clean_sessions["day_offset"].between(-7, -1)
].copy()
assert clean_sessions["session_id"].is_unique, "Conflicting records need a resolution rule."
clean_sessions["minutes_played"] = clean_sessions["minutes_played"].mask(
    clean_sessions["minutes_played"].lt(0)
)

print("raw rows:", len(sessions), "clean historical rows:", len(clean_sessions))
print("unknown durations retained:", clean_sessions["minutes_played"].isna().sum())
display(clean_sessions.head())


raw rows: 4965 clean historical rows: 4905
unknown durations retained: 147


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1


In [5]:
assert clean_sessions["session_id"].is_unique
assert clean_sessions["day_offset"].between(-7, -1).all()
assert clean_sessions["minutes_played"].dropna().ge(0).all()
assert clean_sessions["minutes_played"].isna().any()
print("Historical-session checks passed.")


Historical-session checks passed.


<details>
<summary>Follow-up answers</summary>

**1. Why not use dropna() on the whole table?**

It discards valid evidence of activity and may systematically remove particular users or devices. Keep valid fields and handle missing predictors later.

**2. Why drop exact duplicates rather than silently keep the first session ID?**

The contract permits identical repeated exports. Conflicting rows require a documented resolution rule; arbitrary first-row selection could discard valid information.

**3. Is this cleaning before splitting a leak?**

These fixed validity rules use neither labels nor statistics learned across players. Median imputation, scaling and category learning are different: fit them only on training data.

</details>


## Problem 1 · Step 3 — Aggregate and join (9–14 min)

Build **player_features**, one row for every player. Add **session_count**, **total_minutes**, **avg_minutes**, and **completion_rate** from clean history. Keep a duration-observation count to distinguish unknown durations from inactivity. For a player with no sessions, use 0 for these activity features. For a player with sessions but no valid durations, keep total and average minutes missing.

Report (1) the fraction of players with no historical activity, and (2) average per-player observed total minutes by device. Include inactive players; keep an explicit missing-device group. Counts accompanying the averages should show missing-duration coverage.

**Say before coding:** "I will aggregate before joining, so players with many sessions do not accidentally receive more training weight. A left join preserves inactive players."

**Possible interviewer questions**

1. Why size rather than count for the number of sessions?
2. What happens if you join players directly to all sessions?
3. Does higher play time on one device prove that device increases engagement?


In [6]:
activity = clean_sessions.groupby("user_id").agg(
    session_count=("session_id", "size"),
    total_minutes=("minutes_played", "sum"),
    avg_minutes=("minutes_played", "mean"),
    completion_rate=("completed", "mean"),
    known_duration_count=("minutes_played", "count"),
)
# A sum over entirely missing durations should be unknown, not zero.
activity.loc[activity["known_duration_count"].eq(0), "total_minutes"] = np.nan
player_features = players.merge(
    activity, on="user_id", how="left", validate="one_to_one"
)
no_sessions = player_features["session_count"].isna()
player_features.loc[no_sessions, activity.columns] = 0

print(f"No historical sessions: {no_sessions.mean():.1%}")
device_summary = player_features.groupby("device", dropna=False).agg(
    players=("user_id", "size"),
    players_with_observed_total=("total_minutes", "count"),
    mean_observed_minutes=("total_minutes", "mean"),
)
display(device_summary, player_features.head())
print("Totals sum observed durations; partially missing sessions can understate true play time.")


No historical sessions: 9.3%


,players,players_with_observed_total,mean_observed_minutes
device,,,
console,79,79,174.474937
desktop,279,278,172.055000
mobile,442,441,168.966735
tablet,150,149,167.552416
NaN,50,50,163.201600


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes,session_count,total_minutes,avg_minutes,completion_rate,known_duration_count
0,user_0001,184.0,US,mobile,0,0.00,0.0,0.00,0.000000,0.000000,0.0
1,user_0002,6.0,US,mobile,0,0.00,3.0,NaN,NaN,0.666667,0.0
2,user_0003,56.0,US,desktop,0,0.00,1.0,59.11,59.110000,1.000000,1.0
3,user_0004,39.0,US,desktop,1,247.43,4.0,138.60,34.650000,0.500000,4.0
4,user_0005,295.0,GB,NaN,1,5.96,9.0,342.79,38.087778,0.333333,9.0


Totals sum observed durations; partially missing sessions can understate true play time.


In [1]:
assert len(player_features) == len(players)
assert player_features["user_id"].is_unique
assert player_features["session_count"].notna().all()
assert player_features["session_count"].sum() == len(clean_sessions)
inactive = player_features["session_count"].eq(0)
assert inactive.any()
assert player_features.loc[inactive, ["total_minutes", "avg_minutes"]].eq(0).all().all()
unknown = player_features["session_count"].gt(0) & player_features["known_duration_count"].eq(0)
assert unknown.any()
assert player_features.loc[unknown, ["total_minutes", "avg_minutes"]].isna().all().all()
print("One-row-per-player and missing-versus-zero checks passed.")


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/bytedance/Downloads/Interview (1)/.venv-codesignal/lib/python3.9/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/bytedance/Downloads/Interview (1)/.venv-codesignal/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
  File "/Users/bytedance/Downloads/Interview (1)/.venv-codesignal/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/bytedance/Downloads/Interview (1)/.venv-codesignal/lib/python3.9/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/Users/bytedance/Downloads/Interview (1)

NameError: name 'player_features' is not defined

<details>
<summary>Follow-up answers</summary>

**1. Why size rather than count for the number of sessions?**

size counts rows. Counting minutes_played skips unknown durations and undercounts valid sessions.

**2. What happens if you join players directly to all sessions?**

You get multiple rows per player. That changes the unit of analysis and can leak the same player across a random row split.

**3. Does higher play time on one device prove that device increases engagement?**

No. This is a descriptive association with possible confounding, missing data, and selection effects; it is not a randomized causal comparison.

</details>


## Problem 2 · Step 4 — Select features and split (14–18 min)

Define **numeric_features**, **categorical_features**, and **feature_cols**. Use the five numeric predictors account_age_days, session_count, total_minutes, avg_minutes, completion_rate, plus country and device. Create **X** and **y**, then an 80/20 stratified split: **X_train**, **X_test**, **y_train**, **y_test**. Use SEED. Keep user_id as the row index for auditing, not a predictor.

**Say before coding:** "My feature list is explicit. I will reserve the test set now and fit every learned preprocessing step only on training data."

**Possible interviewer questions**

1. Why use stratify=y?
2. Why is a random split acceptable here, and when is it wrong?
3. Is total_minutes already leaking because we computed it before the split?


In [8]:
numeric_features = [
    "account_age_days", "session_count", "total_minutes", "avg_minutes", "completion_rate"
]
categorical_features = ["country", "device"]
feature_cols = numeric_features + categorical_features

indexed = player_features.set_index("user_id")
X = indexed[feature_cols]
y = indexed["returned_next_week"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print("train:", X_train.shape, "test:", X_test.shape)
print("positive rate:", {"train": y_train.mean(), "test": y_test.mean()})


train: (800, 7) test: (200, 7)
positive rate: {'train': 0.2925, 'test': 0.295}


In [9]:
assert not {"user_id", "returned_next_week", "next_week_minutes"}.intersection(feature_cols)
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(player_features)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert set(y_train) == set(y_test) == {0, 1}
print("Split and feature-boundary checks passed.")


Split and feature-boundary checks passed.


<details>
<summary>Follow-up answers</summary>

**1. Why use stratify=y?**

It approximately preserves class proportions in both subsets. It does not remove temporal leakage or make repeated observations independent.

**2. Why is a random split acceptable here, and when is it wrong?**

This mock assumes independent players at a shared cutoff. To test future deployment, use later cutoffs with mature labels. To test unseen users with repeated rows, use a grouped split; grouping and time answer different evaluation questions.

**3. Is total_minutes already leaking because we computed it before the split?**

No: it uses only each player's own pre-cutoff records, not other players, future activity, or labels. Fitting a global scaler on all players would leak test information.

</details>


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Fit **baseline**, a DummyClassifier using the training-label prior. Then fit **model**, one Pipeline containing a ColumnTransformer and LogisticRegression. Numeric columns: median imputation, then standardization. Categorical columns: most-frequent imputation, then one-hot encoding that tolerates unseen categories. Fit only X_train and y_train. Keep default regularization; set max_iter=500.

**Say before coding:** "A dummy baseline tells me whether I learned more than the class frequency. Logistic regression is fast, interpretable, and a reasonable first model for mixed tabular features."

**Possible interviewer questions**

1. What does Pipeline prevent?
2. Why scale numeric features for logistic regression?
3. What is logistic regression optimizing?


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

#baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=500)),
])
model.fit(X_train, y_train)
print("Both baselines fitted on training players only.")


Both baselines fitted on training players only.


In [11]:
assert list(model.classes_) == [0, 1]
fitted_imputer = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"]
np.testing.assert_allclose(fitted_imputer.statistics_, X_train[numeric_features].median().to_numpy())
print("The numeric imputer learned the training medians.")


The numeric imputer learned the training medians.


<details>
<summary>Follow-up answers</summary>

**1. What does Pipeline prevent?**

At fit, preprocessing learns from training data. At prediction, the same learned transforms are reused. In cross-validation, each fold fits its own preprocessing. It does not fix bad feature timing or an invalid split.

**2. Why scale numeric features for logistic regression?**

It improves optimization and makes the regularization penalty less dependent on arbitrary feature units. Trees generally do not need this scaling.

**3. What is logistic regression optimizing?**

A linear score z = w^T x + b becomes p = 1 / (1 + exp(-z)). Training minimizes binary log loss plus regularization. predict_proba produces probabilities; a threshold produces labels.

</details>


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create **baseline_prob** and **model_prob** for the positive class on X_test. Build a **metrics** table with accuracy, ROC-AUC, average precision (AP), precision, recall and F1. Use a fixed 0.5 threshold for label metrics; pass probabilities to ROC-AUC and AP. Show a confusion matrix with clearly labeled rows and columns. Briefly compare training and test ROC-AUC. Do not tune against the test results.

**Say before coding:** "I will evaluate ranking with probability scores and decision quality at a stated threshold. Accuracy alone may hide poor recall for returning players."

**Possible interviewer questions**

1. Why not compute ROC-AUC using model.predict()? 
2. What do ROC-AUC and AP tell you?
3. What if the business needs 80% recall instead of a 0.5 threshold?


In [ ]:
baseline_prob = baseline.predict_proba(X_test)[:, 1]
model_prob = model.predict_proba(X_test)[:, 1]

def evaluate(y_true, probabilities):
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_true, predicted),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
        "precision@0.5": precision_score(y_true, predicted, zero_division=0),
        "recall@0.5": recall_score(y_true, predicted, zero_division=0),
        "f1@0.5": f1_score(y_true, predicted, zero_division=0),
    }

metrics = pd.DataFrame({
    "prior_baseline": evaluate(y_test, baseline_prob),
    "logistic_regression": evaluate(y_test, model_prob),
}).T
display(metrics.round(3))
display(pd.DataFrame(
    confusion_matrix(y_test, (model_prob >= 0.5).astype(int), labels=[0, 1]),
    index=["actual: no return", "actual: return"],
    columns=["predicted: no return", "predicted: return"],
))
print("train ROC-AUC:", round(roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]), 3))
print("test ROC-AUC:", round(roc_auc_score(y_test, model_prob), 3))


,accuracy,roc_auc,average_precision,precision@0.5,recall@0.5,f1@0.5
prior_baseline,0.705,0.500,0.295,0.000,0.000,0.000
logistic_regression,0.760,0.776,0.640,0.677,0.356,0.467


,predicted: no return,predicted: return
actual: no return,131,10
actual: return,38,21


train ROC-AUC: 0.807
test ROC-AUC: 0.776


In [13]:
assert model_prob.shape == (len(X_test),)
assert np.isfinite(model_prob).all() and ((0 <= model_prob) & (model_prob <= 1)).all()
assert np.isclose(metrics.loc["prior_baseline", "roc_auc"], 0.5)
assert np.isclose(metrics.loc["prior_baseline", "average_precision"], y_test.mean())
print("Probability and baseline-metric checks passed. Model scores are observations, not a pass/fail target.")


Probability and baseline-metric checks passed. Model scores are observations, not a pass/fail target.


<details>
<summary>Follow-up answers</summary>

**1. Why not compute ROC-AUC using model.predict()? **

Hard labels discard score ordering. Use predict_proba[:, 1] for this binary model; check classes_ to verify which column is the positive class.

**2. What do ROC-AUC and AP tell you?**

ROC-AUC measures positive-versus-negative ranking across thresholds. AP summarizes precision-recall performance and its no-skill reference is the positive fraction. AP is not exactly trapezoidal PR-AUC; neither metric establishes calibration.

**3. What if the business needs 80% recall instead of a 0.5 threshold?**

Choose the operating threshold using training cross-validation or a separate validation set, considering precision and cost. Then evaluate the frozen choice once on the untouched test set.

</details>


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

The next cell supplies **new_players** with already-aggregated pre-cutoff features. Compute **new_prob** using the fitted pipeline, without refitting. Include the unknown country and missing attributes. Return the positive-class probability, and give a 30-second summary of your approach, one limitation, and one next experiment.

**Say before coding:** "I will reuse the fitted preprocessing and classifier. The second input deliberately tests missing data and an unseen categorical value."

**Possible interviewer questions**

1. What happens to the unseen country?
2. What would you try next?
3. What must training and serving share?


In [14]:
new_players = pd.DataFrame([
    {"account_age_days": 15, "session_count": 3, "total_minutes": 45,
     "avg_minutes": 15, "completion_rate": 2/3, "country": "US", "device": "mobile"},
    {"account_age_days": np.nan, "session_count": 0, "total_minutes": 0,
     "avg_minutes": 0, "completion_rate": 0, "country": "NEW_COUNTRY", "device": np.nan},
], index=["new_player_A", "new_player_B"])


In [15]:
new_prob = model.predict_proba(new_players[feature_cols])[:, 1]
display(pd.DataFrame({"probability_of_return": new_prob}, index=new_players.index))


,probability_of_return
new_player_A,0.069396
new_player_B,0.008034


In [16]:
assert new_prob.shape == (2,)
assert np.isfinite(new_prob).all() and ((0 <= new_prob) & (new_prob <= 1)).all()
print("Inference handles missing values and an unseen country.")


Inference handles missing values and an unseen country.


<details>
<summary>Follow-up answers</summary>

**1. What happens to the unseen country?**

handle_unknown="ignore" produces all zeros in that categorical feature's one-hot block. The model still uses other features. This avoids a crash, but does not guarantee good predictions for new populations.

**2. What would you try next?**

Use a validation set or cross-validation to compare a tree-based baseline, missingness indicators, and modest feature changes. Evaluate future-cutoff data, calibration and relevant slices before claiming production readiness.

**3. What must training and serving share?**

The prediction cutoff, historical window, deduplication and missing-value rules, feature schema, and fitted preprocessing/model. This prediction cell assumes upstream raw-event aggregation follows the same contract.

</details>

**Example closing statement:** "I built player-level historical features, excluded future information, and trained an imputation/encoding/scaling pipeline with logistic regression. I compared holdout ranking and threshold metrics to a prior baseline. This random split estimates performance on independent players at this cutoff, not robustness over time. My next step would be a chronological validation experiment and a simple tree-based comparison." Insert the scores from your own run; do not invent a business lift.


## If the interviewer extends the problem

1. **Predict next-week minutes instead.** This becomes regression: use next_week_minutes as the target, never as an input. Try a mean baseline and a regularized regressor; compare MAE and RMSE. Discuss the many zeros and long tail. Keep the same feature-time boundary.
2. **Many daily rows per player.** Clarify whether the goal is future predictions for existing players or generalization to unseen players. Use time, group, or a combined split accordingly; mature label windows must not reach into the evaluation period.
3. **A 100 GB session log.** Push filtering/deduplication/aggregation into a database or distributed job. Chunk sums and counts merge easily; an exact distinct-session count needs a deliberate cross-chunk deduplication strategy. Do not average chunk averages without weighting.
4. **A nearly perfect model.** First audit future columns, duplicated entities across subsets, target-derived transformations, and whether preprocessing was fitted on all data. High accuracy is not proof of a good model.

## References and provenance

The interview **format** comes from the description pasted in your [shared conversation](https://chatgpt.com/share/6ad92cab-a9dc-83e8-b72e-654063228d54). The problem, data, code and questions here are original practice material. They are not evidence that Roblox asks this exact task, and this local notebook is not connected to CodeSignal.

Implementation references: [scikit-learn: avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [ROC-AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [pandas merge validation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).
